# Model experimentation: development validation only

Experimental output. Not a clinical decision.

Run with the repository root as the kernel working directory and the pinned dependencies installed. On macOS, XGBoost and LightGBM also need the OpenMP runtime (`brew install libomp`). Notebook code imports only functions from `src/train.py`; reusable orchestration is in `src/experimentation.py` and probability/error helpers are in `src/evaluate.py`.

The existing patient-grouped chronological train/validation/test split is loaded and fingerprinted, never recomputed. No test feature transformation, model prediction, calibration, decision selection, or performance evaluation occurs. Only split-integrity metadata are checked. Earlier full-cohort EDA/hypothesis testing remains a limitation, and validation is reused for development selection rather than presented as an unbiased final estimate.

## Prespecified comparison and tuning protocol

- Compare Logistic Regression, Decision Tree, Random Forest, XGBoost and LightGBM with the same structured preprocessing and frozen validation requests. Reference defaults keep library model settings except documented seed, iteration/estimator counts, deterministic/CPU options, and required multiclass objective. Exact parameters and preprocessing flags are logged.
- For each family, run none, class weights, random oversampling, random undersampling, and SMOTE. The imbalanced-learn pipeline fits preprocessing and then resamples training only. No resampler runs on validation or test. Balanced multiclass sample weights are computed inside each classifier fit, including each CV fit and XGBoost; binary-only positive-class weighting is not substituted.
- Standard SMOTE interpolates already encoded features, including ordinal, binary, one-hot, and (if enabled) text features. Fractional synthetic values can be implausible. This comparison explicitly retains that limitation rather than silently substituting category-aware SMOTE. The text flag remains disabled for the comparable structured experiments.
- Select the strongest default from each model family using the documented development ranking, then take the best two families for tuning. Search uses the selected imbalance strategy; all five strategies were already compared on defaults. The search budget is six seeded configurations per selected family, using RandomizedSearchCV and three expanding chronological training folds. Each fold excludes earlier patients from its validation window and verifies strict time order and no patient overlap. Labels never create or adjust CV time windows.
- The CV refit objective is 0.4 macro F1 + 0.6 mean High/Critical recall, not accuracy. Matching default CV estimates and tuned CV means/SDs are saved alongside validation results. Every candidate/fold fit retains the full classification report, confusion counts, triage errors, probability metrics, parameters and measured fit/score times. Hyperparameter selection never consults the reserved test set.

## Calibration, decision rules and stability

- For each selected family, the better default/tuned **training CV objective** supplies the calibration base. Fit sigmoid and isotonic calibration inside the same forward grouped training folds, with ensemble mode. Earlier prefixes do not cover all training rows as holdouts, so an ordinary ungrouped cross-validation shortcut is not used. Calibration ensembles differ from the single full-training model; a matching uncalibrated fold ensemble isolates the probability-calibration change. The sigmoid/isotonic base ensembles are checked to match before sharing that control. [CalibratedClassifierCV](https://scikit-learn.org/1.6/modules/generated/sklearn.calibration.CalibratedClassifierCV.html).
- Reliability plots compare one-vs-rest mean probability and observed frequency in common equal-width bins; empty bins are omitted and bin support is available in tooltips/CSV. Multiclass Brier is the average **sum** of squared class errors, range zero to two; mean-OVR and per-class Brier and log loss are also retained. Lower Brier alone does not establish calibration, and isotonic can be unstable in sparse bins.
- Evaluate argmax, illustrative cost-sensitive rules, and a fixed class-threshold rule on validation. Cost rows=true class and columns=predicted action. Under-triage costs exceed equal-distance over-triage, with extra penalty for the specified severe errors. Compare predefined under-cost multipliers with an unchanged over-cost and severe surcharge; the threshold variant gives Critical priority above its threshold, then promotes to High above its threshold. These are development scenarios, not clinically elicited utility values. All rules are logged exactly. Probability scores stay unchanged across decision rules applied to the same probabilities.
- Conditional validation uncertainty uses seeded patient-cluster bootstrap resamples, not individual request resampling. These intervals describe fixed predictions, not training/seed variability or post-selection uncertainty. Training CV SDs apply to the **uncalibrated base** configuration; they are not relabelled as calibrated/policy CV stability.
- The shortlist ranking prioritises High/Critical recall and under-triage, penalises over-triage, then incorporates macro F1, Brier and patient-bootstrap recall SD. The exact illustrative weights are printed with the final shortlist. Related calibrated/policy variants share fitted models and are not independent replications.

## Audit and output contract

Frozen split fingerprints, patient separation, chronology, allowed intake features, exact target-copy checks, and pipeline isolation are audited. Near-perfect aggregate validation scores trigger a training-target permutation control and a source/provenance review **before** a shortlist is published. Mechanical checks cannot confirm clinician-label provenance or clinical validity. No high score is presented as a clinical decision.

`artifacts/model_comparison.csv` retains session-scoped rows across reruns, including default fits, each search/default CV fold, calibration base folds, calibrated/control models and decision variants. Complete metrics are also stored in per-run JSON files. Runtime rows include measured fit/score intervals; calibration parent runtimes include their base-fit components, so summing every row would double-count work. Scalar parameters are logged exactly; estimator objects use representations and nonfinite library defaults are represented by strings.

The latest session produces `tuning_comparison.csv`, `model_rankings.csv`, `model_shortlist.csv`, and the written `model_shortlist.md`. `artifacts/experiments/` contains the protocol, CV request IDs, leakage audit, calibration HTML/bin tables, validation probabilities and model registry. Selected fitted defaults/tuned/calibrated models are saved in `models/`. The CSV retains all sessions; the displays below use the latest one.


In [1]:
from src.train import run_model_experimentation, show_model_experimentation

runs, shortlist = run_model_experimentation(
    include_text=False,
    n_iter=6,
    top_candidates=2,
    bootstrap_repeats=200,
)


Completed default_LogisticRegression_none: macro F1=0.720, HC recall=0.679


Completed default_LogisticRegression_class_weights: macro F1=0.689, HC recall=0.703


Completed default_LogisticRegression_random_oversampling: macro F1=0.682, HC recall=0.685


Completed default_LogisticRegression_random_undersampling: macro F1=0.678, HC recall=0.708


Completed default_LogisticRegression_smote: macro F1=0.686, HC recall=0.699


Completed default_DecisionTree_none: macro F1=0.557, HC recall=0.554


... (18 outputs left out)

Completed default_LightGBM_smote: macro F1=0.708, HC recall=0.679


Completed search and calibration for LightGBM


Completed search and calibration for LogisticRegression


Logged 103 runs. Reserved test set not evaluated.


## Computed comparisons, calibration curves and ranked shortlist

The following display reads saved artifacts. Review both recalls, all triage-error directions, macro F1, probability quality, CV variation, and conditional validation uncertainty. Tuning and calibration may worsen held-out development metrics; the tables preserve those outcomes. The final written section gives the ranked shortlist and its trade-offs without making a test-performance claim.


In [2]:
show_model_experimentation()


**Validation only. Experimental output. Not a clinical decision.**

,run_id,model,imbalance_strategy,high_recall,critical_recall,under_triage_rate,severe_under_triage_rate,over_triage_rate,macro_f1,brier_multiclass,runtime_seconds
206,default_LogisticRegression_none,LogisticRegression,none,0.677656,0.680851,0.165394,0.010178,0.104326,0.720223,0.401206,0.329227
207,default_LogisticRegression_class_weights,LogisticRegression,class_weights,0.586081,0.819149,0.137405,0.009330,0.156064,0.688829,0.423678,0.327475
208,default_LogisticRegression_random_oversampling,LogisticRegression,random_oversampling,0.571429,0.797872,0.138253,0.009330,0.157761,0.682244,0.423923,0.493569
209,default_LogisticRegression_random_undersampling,LogisticRegression,random_undersampling,0.586081,0.829787,0.141645,0.011026,0.164546,0.678135,0.428667,0.127977
210,default_LogisticRegression_smote,LogisticRegression,smote,0.589744,0.808511,0.135708,0.009330,0.158609,0.685582,0.425705,0.874869
211,default_DecisionTree_none,DecisionTree,none,0.575092,0.531915,0.207803,0.023749,0.217981,0.556773,0.851569,0.214089
212,default_DecisionTree_class_weights,DecisionTree,class_weights,0.527473,0.606383,0.195929,0.022053,0.217133,0.573750,0.826124,0.234059
213,default_DecisionTree_random_oversampling,DecisionTree,random_oversampling,0.600733,0.542553,0.210348,0.015267,0.211196,0.570580,0.843087,0.303203
214,default_DecisionTree_random_undersampling,DecisionTree,random_undersampling,0.498168,0.734043,0.176421,0.010178,0.258694,0.554364,0.870229,0.083225
215,default_DecisionTree_smote,DecisionTree,smote,0.542125,0.585106,0.201018,0.015267,0.226463,0.563816,0.854962,0.733212


,model,imbalance_strategy,default_run_id,tuned_run_id,best_params,default_cv_macro_f1_mean,default_cv_macro_f1_std,default_cv_hc_recall_mean,default_cv_hc_recall_std,cv_score_scope,cv_macro_f1_mean,cv_macro_f1_std,cv_high_critical_recall_mean,cv_high_critical_recall_std,default_validation_macro_f1,tuned_validation_macro_f1,default_validation_hc_recall,tuned_validation_hc_recall,search_wall_seconds
0,LightGBM,class_weights,default_LightGBM_class_weights,tuned_LightGBM_class_weights,"{""classifier__estimator__reg_lambda"": 1.0, ""cl...",0.669305,0.020807,0.665935,0.031387,Tuned base estimator before calibration/decisi...,0.664479,0.012788,0.682542,0.021295,0.713130,0.680605,0.734101,0.718748,6.409012
1,LogisticRegression,class_weights,default_LogisticRegression_class_weights,tuned_LogisticRegression_class_weights,"{""classifier__estimator__C"": 10.0}",0.662588,0.012140,0.681529,0.012356,Tuned base estimator before calibration/decisi...,0.663653,0.011470,0.684790,0.008550,0.688829,0.687283,0.702615,0.698952,2.339708


,run_id,model,imbalance_strategy,high_recall,critical_recall,under_triage_rate,severe_under_triage_rate,over_triage_rate,macro_f1,brier_multiclass,runtime_seconds
256,uncalibrated_ensemble_LightGBM,LightGBM,class_weights,0.571429,0.808511,0.153520,0.010178,0.168787,0.664528,0.435228,0.021384
257,calibrated_LightGBM_sigmoid,LightGBM,class_weights,0.633700,0.776596,0.156064,0.010178,0.151824,0.687065,0.423796,1.080089
258,calibrated_LightGBM_sigmoid_cost1.5,LightGBM,class_weights,0.655678,0.776596,0.118745,0.004241,0.217133,0.669527,0.423796,0.021974
259,calibrated_LightGBM_sigmoid_cost2,LightGBM,class_weights,0.641026,0.808511,0.106022,0.002545,0.271416,0.635899,0.423796,0.021042
260,calibrated_LightGBM_sigmoid_cost3,LightGBM,class_weights,0.641026,0.808511,0.078880,0.001696,0.360475,0.580821,0.423796,0.021534
261,calibrated_LightGBM_sigmoid_threshold,LightGBM,class_weights,0.582418,0.829787,0.140797,0.006785,0.206955,0.638491,0.423796,0.021218
265,calibrated_LightGBM_isotonic,LightGBM,class_weights,0.670330,0.723404,0.162850,0.011026,0.134012,0.693460,0.420459,1.112970
266,calibrated_LightGBM_isotonic_cost1.5,LightGBM,class_weights,0.659341,0.787234,0.118745,0.003393,0.212892,0.670165,0.420459,0.020865
267,calibrated_LightGBM_isotonic_cost2,LightGBM,class_weights,0.652015,0.808511,0.104326,0.003393,0.270568,0.634902,0.420459,0.022212
268,calibrated_LightGBM_isotonic_cost3,LightGBM,class_weights,0.633700,0.851064,0.081425,0.001696,0.351993,0.583578,0.420459,0.021347


# Ranked development shortlist



Validation-only development ranking; experimental output, not a clinical decision. Many comparisons reused validation, so these are selection results, not unbiased generalisation estimates. The reserved test set was not transformed/evaluated. Earlier full-cohort exploration remains a limitation.



Ranking is prespecified: .30 High recall + .30 Critical recall − .15 under-triage − .30 severe under-triage − .10 over-triage + .20 macro F1 − .10 multiclass Brier − .10 patient-bootstrap High/Critical recall SD. These weights are illustrative development preferences, not elicited clinical utilities.



1. **default_LightGBM_class_weights** — LightGBM, class_weights; High/Critical recall 67.0%/79.8%; under-triage 14.6%, severe 0.8%, over-triage 13.8%; macro F1 0.713; Brier 0.414; patient-bootstrap HC recall SD 0.026, conditional interval [0.682, 0.782]. Forward-CV HC recall SD 0.031 (Default uncalibrated base estimator). Its position reflects the stated recall/error trade-off, followed by F1, probability quality and conditional validation stability.

2. **calibrated_LogisticRegression_isotonic_cost2** — LogisticRegression, class_weights; High/Critical recall 67.8%/83.0%; under-triage 9.2%, severe 0.4%, over-triage 25.6%; macro F1 0.656; Brier 0.414; patient-bootstrap HC recall SD 0.026, conditional interval [0.707, 0.805]. Forward-CV HC recall SD 0.009 (Tuned base estimator before calibration/decision rules). Its position reflects the stated recall/error trade-off, followed by F1, probability quality and conditional validation stability.

3. **default_LightGBM_random_oversampling** — LightGBM, random_oversampling; High/Critical recall 67.8%/76.6%; under-triage 15.0%, severe 1.0%, over-triage 13.5%; macro F1 0.713; Brier 0.415; patient-bootstrap HC recall SD 0.028, conditional interval [0.674, 0.780]. This exact configuration was not forward-CV evaluated; stability here is conditional bootstrap only. Its position reflects the stated recall/error trade-off, followed by F1, probability quality and conditional validation stability.

4. **calibrated_LogisticRegression_sigmoid_threshold** — LogisticRegression, class_weights; High/Critical recall 61.2%/85.1%; under-triage 12.0%, severe 0.8%, over-triage 20.2%; macro F1 0.663; Brier 0.415; patient-bootstrap HC recall SD 0.025, conditional interval [0.685, 0.780]. Forward-CV HC recall SD 0.009 (Tuned base estimator before calibration/decision rules). Its position reflects the stated recall/error trade-off, followed by F1, probability quality and conditional validation stability.

5. **uncalibrated_ensemble_LogisticRegression** — LogisticRegression, class_weights; High/Critical recall 60.1%/83.0%; under-triage 13.3%, severe 0.9%, over-triage 15.8%; macro F1 0.694; Brier 0.427; patient-bootstrap HC recall SD 0.025, conditional interval [0.667, 0.766]. Forward-CV HC recall SD 0.009 (Tuned base estimator before calibration/decision rules). Its position reflects the stated recall/error trade-off, followed by F1, probability quality and conditional validation stability.



## Tuned versus default



- LightGBM: validation macro F1 0.713 default vs 0.681 tuned; HC recall 0.734 vs 0.719. Search selection used the training CV objective; validation improvements are not assumed.

- LogisticRegression: validation macro F1 0.689 default vs 0.687 tuned; HC recall 0.703 vs 0.699. Search selection used the training CV objective; validation improvements are not assumed.



## Selection limits



Grouped forward-CV means/SDs for tuned and matching default candidates are in tuning_comparison.csv. Bootstrap intervals resample validation patients for fixed predictions; they do not measure refit/seed instability or correct for model selection. Related calibrated and decision-rule rows share fitted models and are not independent replications. Cost/threshold decisions leave Brier unchanged because they do not change probabilities. Reported Brier sums squared class errors (range zero to two); per-class and mean-OVR scores are also logged. Sigmoid/isotonic ensembles use only forward-fold training/calibration data; matching uncalibrated ensemble controls isolate the calibration change.

SMOTE interpolates already encoded features, including ordinal and binary values; fractional synthetic values limit its interpretation. Resampling is always inside the fit pipeline. Class weighting uses fold-local multiclass sample weights, including XGBoost; no binary-only scale_pos_weight shortcut.

Isotonic can be unstable in sparse probability bins. Inspect the reliability curves and bin counts rather than interpreting one Brier value as proof of calibration. All costs/thresholds require independent domain validation before use.
